# The US Politician Network — Explainer Notebook

**Computational Social Science · DTU**

This notebook documents every step of our pipeline: from data collection through preprocessing and analysis. It is the technical companion to our [website](https://keremozemre.github.io/Social_computational_science_Project/), which presents the same work for a non-technical reader.

The notebook is structured into 4 sections per the project brief:

1. **Motivation** — what dataset, why, and what we want the reader to take away
2. **Basic stats** — data cleaning, preprocessing, and the resulting dataset shape
3. **Tools, theory and analysis** — text and network methods, how they work, why we used them
4. **Discussion** — what went well, what is still missing

Heavy preprocessing cells (Wikidata SPARQL queries, the Wikipedia article fetch, the tokenisation pass) are wrapped in `if False:` so that running the notebook end-to-end does not re-trigger a 25-minute API job. The cached outputs (`politicians_with_communities.csv`, `politicians_full_text.csv`, `tokenized_wikipage.csv`) are loaded directly. Change `if False:` to `if True:` if you want to actually re-run a preprocessing stage.

# 1 — Motivation

## 1.1 What is your dataset?

Our dataset is a network of **6,138 American politicians** extracted from Wikipedia, plus the full plain-text article for each one of them. Concretely it has three layers:

- **Nodes** — 6,138 US politicians, each tagged with metadata from Wikidata: name, party, gender, position, state, education, career start/end dates, and a derived boolean for whether they attended a prestigious school.
- **Edges** — an undirected link between two politicians whenever their Wikipedia articles link to each other. This gives roughly 19,700 edges.
- **Text** — the full cleaned Wikipedia article for each politician, with boilerplate footer sections (References, See also, External links, Bibliography, etc.) stripped. About 56 million characters in total.

Sources:
- **Wikidata SPARQL endpoint** for the politician list and structured metadata.
- **MediaWiki `extracts` API** for the article text.
- **Wikipedia link graph** for the connections between politicians.

## 1.2 Why did you choose this/these particular dataset(s)?

We chose to build on this dataset, to get a better overview of how, especially in such a polar political world, different political actors are represented and how they interact with each other (connected). Furthermore, the motivation stems from Wikipedia increasingly becoming a source of information for the society. We firstly want to investigate whether, wikipedia in fact offers important information to learn politics or rather does it dwell more on controversies (word clouds). Secondly, we want to determine whether Wikipedia is in fact neutral or does it contain certain bias towards specific groups of individuals, and hence whether it reinforces false information or not.

## 1.3 What was your goal for the end user's experience?

For the final takeaway, we want to both give the user an understanding of how political interactions especially in this day and age function, by highlighting the network statistics and the communities that form the network. But further also to evaluate Wikipedia as a natural and educatory political source.

# 2 — Basic stats. Let's understand the dataset better

## 2.1 Data collection — SPARQL + Wikipedia link walk

We use a three-phase approach:

1. **Seed query** — Query Wikidata for politicians ranked by the number of Wikipedia language editions that cover them (wikipedia cross coverage, an estimate for importance of politicians). Extract the top 1000 seed politicinas that fits the criteria filters.

Filters applied:
- Must be a human (`wd:Q5`).
- Occupation politician (`wd:Q82955`).
- Must have held a US political office (`office wdt:P17 wd:Q30`).
- Birth date ≥ 1990 (so we focus on the modern politicians old enough).
- Death date < 1980 (so we focus don't include historical politicial figures)

2. **Link walking Seed Pages** — For each seed's English Wikipedia page, follow internal links. Politicians that the seeds link to (and that themselves are US politicians) become "neighbour" nodes, and are included in the dataset extending the number of politicians we have to 3776.

Filters applied:
- Must be a human (`wd:Q5`).
- Occupation politician (`wd:Q82955`).
- Must have held a US political office (`office wdt:P17 wd:Q30`).
- Birth date ≥ 1990 (so we focus on the modern politicians old enough).
- Death date < 1980 (so we focus don't include historical politicial figures)

3. **Link Walking Neighbour Pages** — We apply the same as step 2 but now on the neighbours we have just extended to. So we also include all the politicians that are linked to all the neighbour politicians, with the exact same method. This increases our politicians size to 6625.

Filters applied:
- Must be a human (`wd:Q5`).
- Occupation politician (`wd:Q82955`).
- Must have held a US political office (`office wdt:P17 wd:Q30`).
- Birth date ≥ 1990 (so we focus on the modern politicians old enough).
- Death date < 1980 (so we focus don't include historical politicial figures)

4. **Extract Relevant Information** — For each politician, we extract:

- Position  
- Career start and end dates  
- Nationality  
- Birth and death dates  
- Party affiliation  
- Gender  
- Education  
- State  
- Wikipedia article link  

5. **Manual Filtering** — We clean and consolidate the dataset by:

- Merging duplicate entries for the same politician and keeping only their most prestigious position.
- Simplifying party affiliations into three groups: **Liberal**, **Democrat**, and **Other**.
- Estimating political career span using the earliest and latest recorded positions.
- Removing politicians whose careers ended before 1980.

The final dataset, **politicians_network_nodes_filtered.csv**, contains 6,138 politicians and 15 features:

`wikidata_id, name, position, nationality, birthdate, deathdate, party, gender, education, state, wikipedia_url, career_start, career_end, connections, is_seed`

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    import requests
    import time
    import json
    import re
    import pandas as pd
    from collections import defaultdict
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from datetime import datetime, timedelta
    from tqdm import tqdm
    import threading
    import random
    from bs4 import BeautifulSoup


### 1 — Shared session & constants

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    S = requests.Session()
    S.headers.update({"User-Agent": "PoliticiansNetwork/1.0 (kerem.ozemre@icloud.com)"})

    BASE_WIKI  = "https://en.wikipedia.org/w/api.php"
    SPARQL_URL = "https://query.wikidata.org/sparql"

    SKIP_PREFIXES = ("Special:", "Main_Page", "Wikipedia:", "Help:", "File:", "Portal:")


    def get_with_retry(url, params=None, session=S, max_retries=6):
        """GET with exponential backoff on 429 / 5xx."""
        for attempt in range(max_retries):
            try:
                r = session.get(url, params=params, timeout=30)
                if r.status_code == 429:
                    wait = 2 ** attempt + random.uniform(0.5, 1.5)
                    print(f"  Rate-limited, waiting {wait:.1f}s…")
                    time.sleep(wait)
                    continue
                r.raise_for_status()
                return r
            except requests.RequestException as e:
                if attempt == max_retries - 1:
                    raise RuntimeError(f"Failed after {max_retries} retries: {url}") from e
                time.sleep(2 ** attempt)
        raise RuntimeError(f"Failed after {max_retries} retries: {url}")


In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    SESSION = requests.Session()
    SESSION.headers.update({"User-Agent": "WikidataResearchBot/1.0 (kerem.ozemre@icloud.com)"})


    def run_sparql(query, retries=7, base_sleep=2.0):
        for attempt in range(retries):
            try:
                r = SESSION.post(
                    SPARQL_URL,
                    data={"query": query},
                    headers={"Accept": "application/json",
                             "User-Agent": "WikidataResearchBot/1.0"},
                    timeout=120,
                )
                if r.status_code == 429:
                    wait = base_sleep * (2 ** attempt)
                    print(f"  Wikidata 429 — waiting {wait:.0f}s (attempt {attempt+1})")
                    time.sleep(wait)
                    continue
                if r.status_code in (500, 502, 503, 504):   # ← added 504
                    wait = base_sleep * (2 ** attempt)
                    print(f"  Wikidata {r.status_code} — waiting {wait:.0f}s (attempt {attempt+1})")
                    time.sleep(wait)
                    continue
                r.raise_for_status()
                return r.json()
            except requests.exceptions.Timeout:
                wait = base_sleep * (2 ** attempt)
                print(f"  Wikidata timeout (attempt {attempt+1}/{retries}) — waiting {wait:.0f}s")
                time.sleep(wait)
            except Exception as e:
                if attempt == retries - 1:
                    raise RuntimeError(f"SPARQL failed: {e}")
                time.sleep(base_sleep * (2 ** attempt))
        raise RuntimeError(f"SPARQL failed after {retries} retries")


    def chunked(lst, size):
        for i in range(0, len(lst), size):
            yield lst[i:i + size]


    def parse_wikidata_date(raw):
        """Extract YYYY-MM-DD from a Wikidata timestamp string."""
        if not raw:
            return None
        m = re.match(r"[+-]?(\d{4}-\d{2}-\d{2})", raw)
        return m.group(1) if m else None


    def extract_qid(uri):
        return uri.split("/")[-1]


### 3 — Popularity-ranked seed fetcher

Queries Wikidata for politicians ordered by **sitelink count** — the number of Wikipedia
language editions that cover them. This is the best available proxy for global notability
without hitting the pageviews API.

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    def make_seed_query(limit=500, offset=0):
        return f"""
        SELECT DISTINCT ?person (COUNT(?sitelink) AS ?links) WHERE {{
          ?person wdt:P31 wd:Q5 ;       # person
                  wdt:P106 wd:Q82955 . # occupation: politician

          ?person p:P39 ?stmt .         # held position (P39) statement
          ?stmt ps:P39 ?office .        
          ?office wdt:P17 wd:Q30 .          # office belongs to the United States

          OPTIONAL {{ ?person wdt:P569 ?birth. }} # date of birth
          OPTIONAL {{ ?person wdt:P570 ?death. }} # date of death  

          FILTER(
            BOUND(?birth) && YEAR(?birth) <= 1990
            && ( !BOUND(?death) || YEAR(?death) >= 1980 )
          )

          ?sitelink schema:about ?person .
        }}
        GROUP BY ?person
        ORDER BY DESC(?links) # most linked = most cross-wiki-covered
        LIMIT {limit}
        OFFSET {offset}
        """


    def get_popular_politician_ids(max_seeds=1000, page_size=500):
        """
        Return QID URI list for the most cross-wiki-covered politicians,
        paginating until max_seeds is reached or results are exhausted.
        page_size kept at 500 — GROUP BY + ORDER BY queries time out above ~500.
        """
        ids    = []
        offset = 0

        while len(ids) < max_seeds:
            query    = make_seed_query(limit=page_size, offset=offset)
            data     = run_sparql(query)
            bindings = data["results"]["bindings"]
            if not bindings:
                break
            batch = [b["person"]["value"] for b in bindings]
            ids.extend(batch)
            print(f"  Collected {len(ids)} seeds…")
            offset += page_size
            time.sleep(2)   # GROUP BY queries are heavier — be polite

        return ids[:max_seeds]


### 4 — Label fetcher

Resolves QIDs → English Wikipedia article titles.
The link-walker needs these titles to look up pages.
Also used to refresh titles for neighbor nodes discovered in expansion passes.


In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    def fetch_labels(qids, batch_size=400):
        """
        Fetch English Wikipedia titles for a list of QIDs.
        Returns {wikipedia_title: qid} for QIDs that have an enwiki article.
        """
        name_to_qid = {}

        for chunk in tqdm(list(chunked(qids, batch_size)), desc="Fetching labels"):
            values = " ".join(f"wd:{q}" for q in chunk)
            query  = f"""
            SELECT ?person ?article WHERE {{
              VALUES ?person {{ {values} }}
              ?article schema:about ?person ;
                       schema:isPartOf <https://en.wikipedia.org/> .
            }}
            """
            data = run_sparql(query)
            for b in data["results"]["bindings"]:
                qid   = b["person"]["value"].split("/")[-1]
                title = (b["article"]["value"]
                         .replace("https://en.wikipedia.org/wiki/", "")
                         .replace("_", " "))
                name_to_qid[title] = qid
            time.sleep(1)

        print(f"  {len(name_to_qid)} QIDs have an English Wikipedia article")
        return name_to_qid


### 5 — Wikipedia link-walker (degree computation)

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    # ── Rate-limit governor ────────────────────────────────────────────────────────
    _WIKI_SEM      = threading.Semaphore(4)
    _BACKOFF_LOCK  = threading.Lock()
    _BACKOFF_UNTIL = 0.0

    def _wait_for_backoff():
        with _BACKOFF_LOCK:
            target = _BACKOFF_UNTIL
        remaining = target - time.monotonic()
        if remaining > 0:
            time.sleep(remaining)

    def _set_backoff(seconds):
        global _BACKOFF_UNTIL
        with _BACKOFF_LOCK:
            _BACKOFF_UNTIL = max(_BACKOFF_UNTIL, time.monotonic() + seconds)


    # ── Fetch links from article body only (no navboxes/tables) ───────────────────
    def get_page_links_body_only(title):
        """
        Fetch only links that appear in article body prose.
        Strips navboxes, delegation tables, infoboxes — these are the source
        of the CA-delegation cluster artifact (e.g. every CA rep links to every
        other CA rep via the 'California delegation' navbox on their page).
        """
        session = requests.Session()
        session.headers.update({"User-Agent": "PoliticiansNetwork/1.0 (kerem.ozemre@icloud.com)"})

        params = {
            "action":    "parse",
            "page":      title,
            "prop":      "text",
            "format":    "json",
            "redirects": 1,
        }

        for attempt in range(6):
            try:
                r = get_with_retry("https://en.wikipedia.org/w/api.php",
                                   params=params, session=session)
                html = r.json().get("parse", {}).get("text", {}).get("*", "")
                break
            except Exception as e:
                if attempt == 5:
                    return []
                time.sleep(2 ** attempt)

        soup = BeautifulSoup(html, "html.parser")

        # Remove navboxes, tables, infoboxes, succession boxes, references
        for tag in soup.find_all(class_=[
            "navbox", "navbox-inner", "navbox-subgroup", "navbox-list",
            "wikitable", "infobox", "succession-box",
            "mw-references-wrap", "reflist", "portal",
            "noprint", "side-box", "sistersitebox",
        ]):
            tag.decompose()

        # Remove all <table> tags (catches delegation tables not covered above)
        for tag in soup.find_all("table"):
            tag.decompose()

        # Remove reference/external sections
        for section_id in ["References", "External_links", "See_also", "Further_reading"]:
            tag = soup.find(id=section_id)
            if tag:
                # Remove everything from this heading onward
                for sibling in tag.find_all_next():
                    sibling.decompose()
                tag.decompose()

        # Collect links only from <p> paragraph tags
        links = []
        for p in soup.find_all("p"):
            for a in p.find_all("a", href=True):
                href = a["href"]
                if href.startswith("/wiki/") and ":" not in href:
                    links.append(href[6:].replace("_", " "))

        return links


    # ── Thread-safe caches ─────────────────────────────────────────────────────────
    _title_lock = threading.Lock()
    _qid_lock   = threading.Lock()

    def resolve_titles_to_qids_cached(titles, cache, batch_size=50):
        with _title_lock:
            uncached = [t for t in titles if t not in cache]
        for i in range(0, len(uncached), batch_size):
            batch  = uncached[i:i + batch_size]
            params = {
                "action": "query", "titles": "|".join(batch),
                "prop": "pageprops", "ppprop": "wikibase_item", "format": "json",
            }
            r     = get_with_retry(BASE_WIKI, params=params)
            pages = r.json()["query"]["pages"]
            norm  = {n["from"]: n["to"] for n in r.json()["query"].get("normalized", [])}
            result = {}
            for page in pages.values():
                qid    = page.get("pageprops", {}).get("wikibase_item")
                ptitle = page.get("title", "")
                result[ptitle] = qid
            for orig, norm_title in norm.items():
                if norm_title in result:
                    result[orig] = result[norm_title]
            with _title_lock:
                cache.update(result)
            time.sleep(0.1)
        with _title_lock:
            return {t: cache.get(t) for t in titles}


    def filter_politician_qids_cached(qids, cache, batch_size=200):
        with _qid_lock:
            unchecked = [q for q in qids if q not in cache["seen"]]
        for i in range(0, len(unchecked), batch_size):
            batch  = unchecked[i:i + batch_size]
            values = " ".join(f"wd:{q}" for q in batch)
            query  = f"""
            SELECT ?person WHERE {{
              VALUES ?person {{ {values} }}
              ?person wdt:P106 wd:Q82955 .

              # Must have held a US office
              ?person p:P39 ?stmt .
              ?stmt ps:P39 ?office .
              ?office wdt:P17 wd:Q30 .

              # Same date filter as seeds
              OPTIONAL {{ ?person wdt:P569 ?birth. }}
              OPTIONAL {{ ?person wdt:P570 ?death. }}
              FILTER(
                BOUND(?birth) && YEAR(?birth) <= 1990
                && ( !BOUND(?death) || YEAR(?death) >= 1980 )
              )
            }}
            """
            data  = run_sparql(query)
            found = [item["person"]["value"].split("/")[-1]
                     for item in data["results"]["bindings"]]
            with _qid_lock:
                cache["politicians"].update(found)
                cache["seen"].update(batch)
            time.sleep(0.5)
        with _qid_lock:
            return cache["politicians"]


    # ── Serial walker with adaptive throttle ──────────────────────────────────────
    def compute_politician_degrees(name_to_qid, title_qid_cache=None, politician_cache=None):
        """
        Walk Wikipedia pages for each politician in name_to_qid.
        Returns (degrees, neighbours) dicts keyed by QID.
        Caches are shared across calls so repeated QIDs are only looked up once.
        """
        if title_qid_cache is None:
            title_qid_cache = {}
        if politician_cache is None:
            politician_cache = {"seen": set(), "politicians": set()}

        degrees   = {}
        neighbors = {}
        delay     = 0.5   # slightly higher base delay since parse API is heavier than links API

        for idx, (title, qid) in enumerate(tqdm(name_to_qid.items(), desc="Walking pages"), 1):
            linked_titles = []
            for attempt in range(6):
                try:
                    linked_titles = get_page_links_body_only(title)
                    delay = max(0.3, delay * 0.97)
                    break
                except Exception as e:
                    if "429" in str(e):
                        delay = min(30, delay * 2)
                        print(f"  429 on '{title}' — slowing to {delay:.1f}s")
                        time.sleep(delay)
                    elif attempt == 5:
                        print(f"  Giving up on '{title}': {e}")
                        break
                    else:
                        time.sleep(2 ** attempt)

            if not linked_titles:
                degrees[qid]   = 0
                neighbors[qid] = []
            else:
                t2q            = resolve_titles_to_qids_cached(linked_titles, title_qid_cache)
                linked_qids    = [q for q in t2q.values() if q]
                pol_set        = filter_politician_qids_cached(linked_qids, politician_cache) if linked_qids else set()
                pol_qids       = [q for q in linked_qids if q in pol_set]
                degrees[qid]   = len(pol_qids)
                neighbors[qid] = pol_qids

            time.sleep(delay)

            if idx % 50 == 0:
                print(f"  [{idx}/{len(name_to_qid)}] delay={delay:.2f}s | "
                      f"title cache={len(title_qid_cache)} | "
                      f"qid cache={len(politician_cache['seen'])}")

        return degrees,neighbors


### 6 — Feature fetcher

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    def make_details_query(person_ids):
        values = " ".join(f"wd:{p.split('/')[-1]}" for p in person_ids)
        return f"""
        SELECT
          ?person ?personLabel ?positionLabel
          ?start ?end ?natLabel ?birth ?death
          ?partyLabel ?genderLabel ?educationLabel ?stateLabel ?article
        WHERE {{
          VALUES ?person {{ {values} }}

          ?person p:P39 ?statement .
          ?statement ps:P39 ?position .
          OPTIONAL {{ ?statement pq:P580 ?start. }}
          OPTIONAL {{ ?statement pq:P582 ?end. }}

          OPTIONAL {{ ?person wdt:P27 ?nat. }}
          OPTIONAL {{ ?person wdt:P569 ?birth. }}
          OPTIONAL {{ ?person wdt:P570 ?death. }}
          OPTIONAL {{ ?person wdt:P102 ?party. }}
          OPTIONAL {{ ?person wdt:P21 ?gender. }}
          OPTIONAL {{ ?person wdt:P69 ?education. }}
          OPTIONAL {{ ?position wdt:P131 ?state. }}
          OPTIONAL {{
            ?article schema:about ?person ;
                     schema:isPartOf <https://en.wikipedia.org/> .
          }}

          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
        """


    def fetch_details(person_ids, batch_size=50):
        rows = []
        for i, chunk in enumerate(chunked(person_ids, batch_size), 1):
            print(f"  Details batch {i}/{-(-len(person_ids)//batch_size)} ({len(chunk)} people)")
            data = run_sparql(make_details_query(chunk))
            for item in data["results"]["bindings"]:
                rows.append({
                    "wikidata":      extract_qid(item["person"]["value"]),
                    "name":          item.get("personLabel",   {}).get("value"),
                    "position":      item.get("positionLabel", {}).get("value"),
                    "start":         parse_wikidata_date(item.get("start",  {}).get("value")),
                    "end":           parse_wikidata_date(item.get("end",    {}).get("value")),
                    "nationality":   item.get("natLabel",      {}).get("value"),
                    "birth_date":    parse_wikidata_date(item.get("birth",  {}).get("value")),
                    "death_date":    parse_wikidata_date(item.get("death",  {}).get("value")),
                    "party":         item.get("partyLabel",    {}).get("value"),
                    "gender":        item.get("genderLabel",   {}).get("value"),
                    "education":     item.get("educationLabel",{}).get("value"),
                    "state":         item.get("stateLabel",    {}).get("value"),
                    "wikipedia_url": item.get("article",       {}).get("value"),
                })
            time.sleep(1.5)
        return rows


### 7 — Manual Filtering

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    POSITION_RANK = {
        "President of the United States": 1,
        "Vice President of the United States": 2,
        "Secretary of State": 3,
        "Secretary of Defense": 4,
        "Attorney General of the United States": 5,
        "Secretary of the Treasury": 6,
        "White House Chief of Staff": 7,
        "United States Senator": 8,
        "Speaker of the House of Representatives": 9,
        "House Majority Leader": 10,
        "Senate Majority Leader": 10,
        "United States Representative": 11,
        "Governor": 12,
        "Ambassador": 13,
        "Lieutenant Governor": 14,
        "State Senator": 15,
        "State Representative": 16,
        "Mayor": 17,
    }

    def position_priority(pos):
        if pos is None:
            return 999
        if pos in POSITION_RANK:
            return POSITION_RANK[pos]
        for key, rank in POSITION_RANK.items():
            if key.lower() in pos.lower():
                return rank
        return 998
    def simplify_party(p):
        if pd.isna(p):
            return "Other"

        p_lower = p.lower()

        # Major parties
        if "democratic" in p_lower:
            return "Democrat"
        if "republican" in p_lower:
            return "Republican" 
        return "Other"





    def dedup_df(df):
        """
        Per unique wikidata QID:
          - position  → most prestigious title (lowest POSITION_RANK score)
          - start     → earliest start date across all positions
          - end       → latest end date across all positions
          - all other columns (party, gender, education, etc.) → from the most prestigious row
        """
        df = df.copy()
        df["position_rank"] = df["position"].apply(position_priority)

        # ── Most prestigious row per person (for title + all metadata) ─────────────
        best = (df.sort_values("position_rank")
                  .drop_duplicates(subset=["wikidata"], keep="first")
                  .set_index("wikidata"))

        # ── Earliest start date per person ────────────────────────────────────────
        earliest_start = (df[df["start"].notna()]
                          .sort_values("start")
                          .drop_duplicates(subset=["wikidata"], keep="first")
                          .set_index("wikidata")["start"])

        # ── Latest end date per person ─────────────────────────────────────────────
        # Treat NaN end as still in office — represented as None, not overwritten
        latest_end = (df[df["end"].notna()]
                      .sort_values("end", ascending=False)
                      .drop_duplicates(subset=["wikidata"], keep="first")
                      .set_index("wikidata")["end"])

        # ── Assemble ───────────────────────────────────────────────────────────────
        best["career_start"] = best.index.map(earliest_start)
        best["career_end"]   = best.index.map(latest_end)

        # If someone has no end date at all, career_end stays None (still in office)
        # If someone has no start date at all, career_start stays None

        result = best.drop(columns=["start", "end", "position_rank"]).reset_index()

        

        print(f"  {len(result)} unique politicians after dedup")
        print(f"  career_start filled: {result['career_start'].notna().sum()}")
        print(f"    career_end filled: {result['career_end'].notna().sum()} "
              f"({result['career_end'].isna().sum()} still in office or unknown)")
        return result

### 8 — End-to-end pipeline


In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    def run_pipeline(max_seeds=1000, expand_neighbors=False):
        # ── Phase 1: popularity-ranked seeds ──────────────────────────────────────
        print("=" * 60)
        print("Phase 1 — Fetching popularity-ranked seed IDs…")
        seed_uris = get_popular_politician_ids(max_seeds=max_seeds)
        seed_qids = {extract_qid(u) for u in seed_uris}
        print(f"  Unique seeds: {len(seed_qids)}")

        print("\nFetching Wikipedia titles for seeds…")
        seed_titles = fetch_labels(list(seed_qids))
        print(f"  Seeds with enwiki article: {len(seed_titles)}")

        # Shared caches — reused across both expansion passes
        title_qid_cache  = {}
        politician_cache = {"seen": set(), "politicians": set()}

        # ── Phase 2: link-walk seeds ───────────────────────────────────────────────
        print("\n" + "=" * 60)
        print("Phase 2 — Link-walking seed pages…")
        degrees, neighbours = compute_politician_degrees(
            seed_titles, title_qid_cache, politician_cache
        )

        neighbor_qids_1 = {q for conns in neighbours.values() for q in conns} - seed_qids
        all_qids        = seed_qids | neighbor_qids_1
        print(f"  Seeds              : {len(seed_qids)}")
        print(f"  New from pass 1    : {len(neighbor_qids_1)}")
        print(f"  Total after pass 1 : {len(all_qids)}")

        # ── Phase 3: optional second expansion pass ────────────────────────────────
        if expand_neighbors and neighbor_qids_1:
            print("\n" + "=" * 60)
            print("Phase 3 — Link-walking neighbor pages…")
            neighbor_titles = fetch_labels(list(neighbor_qids_1))
            degrees_2, neighbours_2 = compute_politician_degrees(
                neighbor_titles, title_qid_cache, politician_cache
            )

            neighbor_qids_2 = {q for conns in neighbours_2.values() for q in conns} - all_qids
            all_qids       |= neighbor_qids_2
            print(f"  New from pass 2    : {len(neighbor_qids_2)}")
            print(f"  Total after pass 2 : {len(all_qids)}")

            degrees.update(degrees_2)
            neighbours.update(neighbours_2)

        # ── Link-walk the non-seed neighbors too ──────────────────────────────────
        # Seeds already have degrees/neighbours from the walk above.
        # Non-seed nodes were only *discovered* as neighbors — we haven't walked
        # their pages yet, so we do that now to get their connections too.
        non_seed_qids   = all_qids - seed_qids
        non_seed_titles = fetch_labels(list(non_seed_qids))
        print(f"\nLink-walking {len(non_seed_titles)} non-seed pages for their connections…")
        degrees_ns, neighbours_ns = compute_politician_degrees(
            non_seed_titles, title_qid_cache, politician_cache
        )
        # Only add — don't overwrite seeds that were already walked
        for qid, deg in degrees_ns.items():
            if qid not in degrees:
                degrees[qid]   = deg
                neighbours[qid] = neighbours_ns[qid]

        # ── Phase 4: fetch features for everyone ──────────────────────────────────
        print("\n" + "=" * 60)
        print(f"Phase 4 — Fetching features for all {len(all_qids)} politicians…")
        all_uris = [f"http://www.wikidata.org/entity/{q}" for q in all_qids]
        rows     = fetch_details(all_uris)

        df = pd.DataFrame(rows)
        df = dedup_df(df)
        df["connections"] = df["wikidata"].map(neighbours).apply(
            lambda x: x if isinstance(x, list) else []
        )
        df["is_seed"] = df["wikidata"].isin(seed_qids)

        # ── Save edges to JSON ─────────────────────────────────────────────────────
        import json
        node_set     = set(df["wikidata"])
        clean_edges  = {
            qid: [n for n in conns if n in node_set]
            for qid, conns in neighbours.items()
            if qid in node_set
        }
        print(f"Saved → politician_edges.json  "
              f"({len(clean_edges)} nodes, "
              f"{sum(len(v) for v in clean_edges.values())} total edge entries)")

        # ── Save node CSV ──────────────────────────────────────────────────────────
        df.to_csv("politicians_network_nodes.csv", index=False)
        print(f"Saved → politicians_network_nodes.csv  ({len(df)} rows)")

        n_seeds = df["is_seed"].sum()
        print(f"\n{'=' * 60}")
        print(f"Pipeline complete.")
        print(f"  Seed nodes     : {n_seeds}")
        print(f"  Expanded nodes : {len(df) - n_seeds}")
        print(f"  Total nodes    : {len(df)}")
        return df


    df = run_pipeline(max_seeds=1000, expand_neighbors=True)


### 9 — Save

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    if not df.empty:
        print(f"Shape: {df.shape}")
        print(df.head(20).to_string())
    else:
        print("Nothing to save — DataFrame is empty.")


In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    df["party"] = df["party"].apply(simplify_party)

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    career_end_dt = pd.to_datetime(df["career_end"], errors="coerce")

    # Keep if: career_end is NaT (still active / unknown) OR career ended >= 1980
    mask = career_end_dt.isna() | (career_end_dt >= pd.Timestamp("1980-01-01"))

    df = df[mask].reset_index(drop=True)

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    df.to_csv("politicians_network_nodes_filtered.csv", index=False)

## 2.2 Full Wikipedia text extraction

For each politician we fetch their full plain-text Wikipedia article via the MediaWiki `extracts` API with `explaintext=1` and `exsectionformat=wiki` (so `== Section ==` markers are preserved). Then we strip boilerplate footer sectionS: References, Notes, Footnotes, Citations, See also, External links, Bibliography, Sources, Further reading, Works, Publications — and any of their sub-sections.

We do **not** truncate by character count (the API supports this via `exchars` but we want the whole article for sentiment and TF-IDF).

Each article is cached to `wiki_full_text_cache/<wikidata>.json` so the script is resumable on crashes. failed entries are automatically retried on re-run.

_HER SKAL VI SVARE PÅ: why you chose this particular cleaning strategy (e.g. why the footer sections rather than something more aggressive)._

Extracts the full plain-text Wikipedia article for every politician in `politicians_with_communities.csv`.

Output: `politicians_full_text.csv` with the original metadata columns plus one text column `body_text` containing the full cleaned article (intro + all kept sections, with `== Section ==` markers preserved). Boilerplate footer sections (References, See also, External links, Bibliography, Sources, Further reading, Notes, Footnotes, Citations, Works, Publications) are dropped together with their subsections.

Final columns:
`wikidata, name, position, start, end, nationality, birth_date, death_date, party, party_simple, gender, education, state, degree, wikipedia_url, body_text`

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    import os, re, json, time, glob
    from urllib.parse import unquote
    import requests
    import pandas as pd
    from tqdm import tqdm

    API_URL = 'https://en.wikipedia.org/w/api.php'
    CACHE_DIR = 'wiki_full_text_cache'
    INPUT_CSV = 'politicians_with_communities.csv'
    OUTPUT_CSV = 'politicians_full_text.csv'
    REQUEST_DELAY_S = 0.15
    REQUEST_TIMEOUT_S = 30

    os.makedirs(CACHE_DIR, exist_ok=True)

    session = requests.Session()
    session.headers.update({'User-Agent': 'PoliticiansNetwork/1.0 (s245290@dtu.dk) full-article extraction'})

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    df_nodes = pd.read_csv(INPUT_CSV)
    print(f'Loaded {len(df_nodes)} politicians from {INPUT_CSV}')
    print('Columns:', list(df_nodes.columns))
    df_nodes.head(3)

### API helper

Uses `prop=extracts` with `explaintext=1` (clean plain text, no HTML/refs/infoboxes) and `exsectionformat=wiki` (preserves `== Section ==` markers). One title per call so we get full text (batching forces intro-only).

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    def title_from_url(url):
        if not isinstance(url, str) or '/wiki/' not in url:
            return ''
        return unquote(url.rstrip('/').split('/wiki/')[-1])

    def fetch_full_article(title):
        if not title:
            return ''
        params = {
            'action': 'query',
            'format': 'json',
            'titles': title,
            'prop': 'extracts',
            'explaintext': 1,
            'exsectionformat': 'wiki',
            'redirects': 1,
        }
        r = session.get(API_URL, params=params, timeout=REQUEST_TIMEOUT_S)
        r.raise_for_status()
        pages = r.json().get('query', {}).get('pages', {})
        if not pages:
            return ''
        page = next(iter(pages.values()))
        return page.get('extract', '') or ''

### Boilerplate filter

`clean_full_text` removes top-level sections (and their subsections) matching: References, Notes, Footnotes, Citations, See also, External links, Bibliography, Sources, Further reading, Works, Selected works, Publications, Selected publications. Returns the full cleaned article as one string with `== Section ==` markers preserved.

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    SECTION_HEADER_RE = re.compile(r'^(=+)\s*(.+?)\s*\1\s*$', re.MULTILINE)

    BOILERPLATE_PATTERNS = [
        'references', 'notes', 'footnotes', 'citations',
        'see also', 'external links',
        'bibliography', 'sources', 'further reading',
        'works', 'selected works', 'publications', 'selected publications',
    ]

    def is_boilerplate(section_title):
        t = section_title.strip().lower().rstrip(':')
        if t in BOILERPLATE_PATTERNS:
            return True
        for p in BOILERPLATE_PATTERNS:
            if t == p or t.startswith(p + ' ') or t.endswith(' ' + p):
                return True
        return False

    def clean_full_text(full_text):
        """Return the full article as one string with boilerplate sections removed.
        Keeps the intro plus all non-boilerplate sections, with `== Section ==` headers preserved."""
        if not full_text:
            return ''
        matches = list(SECTION_HEADER_RE.finditer(full_text))
        if not matches:
            return full_text.strip()
        intro = full_text[:matches[0].start()].strip()
        sections = []
        for i, m in enumerate(matches):
            level = len(m.group(1))
            title = m.group(2).strip()
            s = m.end()
            e = matches[i+1].start() if i+1 < len(matches) else len(full_text)
            sections.append((level, title, full_text[s:e].strip()))
        cleaned_parts = []
        if intro:
            cleaned_parts.append(intro)
        skipping = False
        for level, title, body in sections:
            if level == 2:
                skipping = is_boilerplate(title)
            if skipping:
                continue
            marker = '=' * level
            cleaned_parts.append(f'{marker} {title} {marker}')
            if body:
                cleaned_parts.append(body)
        return '\n\n'.join(cleaned_parts).strip()

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    _demo = (
        'Barack Obama is a politician.\n\n'
        '== Early life and education ==\n\nBorn in Honolulu.\n\n'
        '=== Childhood ===\n\nHawaii.\n\n'
        '== Political career ==\n\nServed as senator.\n\n'
        '== References ==\n\n[1] cite\n\n'
        '=== Sub-ref ===\n\nshould drop\n\n'
        '== External links ==\n\nlinks\n'
    )
    print(clean_full_text(_demo))

### Cache helpers — successful entries reused, errored entries retried

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    def cache_path(wikidata_id):
        return os.path.join(CACHE_DIR, f'{wikidata_id}.json')

    def load_cached(wikidata_id):
        p = cache_path(wikidata_id)
        if os.path.exists(p):
            try:
                with open(p, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except Exception:
                return None
        return None

    def save_cached(wikidata_id, payload):
        with open(cache_path(wikidata_id), 'w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False)

    def is_usable_cache(payload):
        return payload is not None and not payload.get('error')

    def get_or_fetch(wikidata_id, title):
        cached = load_cached(wikidata_id)
        if is_usable_cache(cached):
            return cached, True
        try:
            text = fetch_full_article(title)
            payload = {'wikidata': wikidata_id, 'title': title, 'raw_text': text, 'error': None}
        except Exception as e:
            payload = {'wikidata': wikidata_id, 'title': title, 'raw_text': '', 'error': str(e)}
        save_cached(wikidata_id, payload)
        return payload, False

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    _existing = glob.glob(os.path.join(CACHE_DIR, '*.json'))
    _err = 0
    _ok = 0
    for f in _existing:
        try:
            d = json.load(open(f, encoding='utf-8'))
            if d.get('error'):
                _err += 1
            else:
                _ok += 1
        except Exception:
            pass
    print(f'Cache files: {len(_existing)} ({_ok} OK, {_err} errored — will be retried)')

### Main extraction loop

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    rows = []
    errors = []
    fetched_from_api = 0
    loaded_from_cache = 0

    subset = df_nodes[df_nodes['wikipedia_url'].notna() & (df_nodes['wikipedia_url'] != '')].copy()
    print(f'Processing {len(subset)} politicians.')

    for _, row in tqdm(subset.iterrows(), total=len(subset), desc='Fetching full articles'):
        wikidata_id = row['wikidata']
        title = title_from_url(row['wikipedia_url'])
        payload, was_cached = get_or_fetch(wikidata_id, title)
        if was_cached:
            loaded_from_cache += 1
        else:
            fetched_from_api += 1
            time.sleep(REQUEST_DELAY_S)
        if payload.get('error'):
            errors.append((wikidata_id, title, payload['error']))
        body_text = clean_full_text(payload.get('raw_text', ''))
        rows.append({'wikidata': wikidata_id, 'body_text': body_text})

    print(f'\nDone. From cache: {loaded_from_cache} | from API: {fetched_from_api} | errors: {len(errors)}')
    if errors:
        print('First 10 errors:')
        for e in errors[:10]:
            print(' ', e[0], e[1], '->', e[2][:120])

### Build wide CSV — metadata + body_text

Renames `career_start` → `start` and `career_end` → `end`.

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    df_text = pd.DataFrame(rows)
    df_meta = df_nodes.rename(columns={'career_start': 'start', 'career_end': 'end'}).copy()
    df_out = df_meta.merge(df_text, on='wikidata', how='left')


    df_out.to_csv(OUTPUT_CSV, index=False)
    print(f'Saved {len(df_out)} rows to {OUTPUT_CSV}')
    print('Columns:', list(df_out.columns))
    df_out.head(3)

### Sanity checks

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    body_len = df_out['body_text'].fillna('').astype(str).str.len()

    print('=== Coverage ===')
    print(f'Rows total              : {len(df_out)}')
    print(f'Empty body_text         : {(body_len == 0).sum()}')
    print()
    print('=== body_text length distribution (chars) ===')
    print(body_len.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(0))
    print()
    print('=== Top 5 longest ===')
    tmp = df_out.assign(_len=body_len)
    print(tmp.nlargest(5, '_len')[['name', '_len']].to_string(index=False))
    print()
    print('=== 5 shortest non-empty ===')
    ne = tmp[tmp['_len'] > 0]
    print(ne.nsmallest(5, '_len')[['name', '_len']].to_string(index=False))
    print()
    print('=== Spot check: did Obama, Biden, Trump come back? ===')
    for q in ['Q76', 'Q6279', 'Q22686']:
        sub = df_out[df_out['wikidata'] == q]
        if len(sub):
            r = sub.iloc[0]
            print(f"  {r['name']:20s} body_len={len(str(r['body_text'])):>7d}")

## 2.3 Tokenisation

Our text-preprocessing pipeline:

1. **Lowercase** the entire article.
2. **Strip URLs and digits** via regex (`http\S+|www\S+` and `\d+`).
3. **Strip non-alphabetic characters** (`[^a-z\s]`) and collapse runs of whitespace.
4. **Remove NLTK English stopwords**.
5. **Lemmatise** with WordNet, using NLTK's POS tagger to pick the right wordnet category (verb/noun/adj/adv).
6. **Detect significant bigrams** via a χ² test on co-occurrence counts (p < 0.001, count > 50). These are collocations like *supreme court* or *vice president*.
7. **Merge collocations** using NLTK's `MWETokenizer`, then lemmatise the remaining single-word tokens.

_Write here why you chose Porter stemming initially but then switched to WordNet lemmatisation (or why you kept both)._

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    import nltk
    import pandas as pd
    import ast

    df = pd.read_csv('/Users/keremozemre/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/social compute/Social_computational_science_Project/politicians_full_text.csv')

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    import nltk
    import re
    nltk.download('punkt_tab')
    from nltk.tokenize import sent_tokenize
    from nltk.stem import PorterStemmer
    from nltk.corpus import stopwords
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words('english'))

    def tokenize(text):
        text = text.lower()
        
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        
        words = text.split()
        filtered_words = [word for word in words if word not in stop_words]
        stems = [stemmer.stem(word) for word in filtered_words]
        
        
        return stems

    df["tokens"] = df["body_text"].dropna().apply(tokenize)

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    df["tokens"]

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    import re
    import math
    import numpy as np
    from collections import Counter
    from scipy.stats import chi2
    from nltk.stem import WordNetLemmatizer
    from nltk.corpus import stopwords, wordnet
    import nltk

    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    def get_wordnet_pos(word):
        tag = nltk.pos_tag([word])[0][1][0].upper()
        tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'R': wordnet.ADV}
        return tag_map.get(tag, wordnet.NOUN)

    def tokenize_lemmatized(abstract):
        """Clean → remove stopwords → lemmatize. Used for bigram discovery."""
        if not isinstance(abstract, str):
            return []
        text = abstract.lower()
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        words = [w for w in text.split() if w not in stop_words]
        return [lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in words]

    df["tokens_lemmatized"] = df["body_text"].apply(tokenize_lemmatized)
    df[["tokens", "tokens_lemmatized"]].head()

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    from nltk.probability import FreqDist

    all_tokens = df["tokens_lemmatized"].explode().tolist()

    fq = FreqDist(all_tokens)
    fq.most_common(10)

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    clean_tokens = [t for t in all_tokens if t is not None and not (isinstance(t, float) and math.isnan(t))]

    bigrams = list(nltk.bigrams(clean_tokens))

    bigram_counts = Counter(bigrams)
    w1_counts     = Counter([w1 for w1, _ in bigrams])
    w2_counts     = Counter([w2 for _, w2 in bigrams])
    N             = len(bigrams)

    chi2_scores  = {}
    p_values     = {}
    co_occurences = {}

    for (w1, w2), nii in bigram_counts.items():
        nio = w1_counts[w1] - nii
        noi = w2_counts[w2] - nii
        noo = N - (nii + nio + noi)

        observed = np.array([[nii, nio], [noi, noo]])
        row_sums  = observed.sum(axis=1)
        col_sums  = observed.sum(axis=0)
        expected  = np.outer(row_sums, col_sums) / N

        chi2_stat = ((observed - expected) ** 2 / expected).sum()
        p_val     = chi2.sf(chi2_stat, df=1)

        co_occurences[(w1, w2)] = nii
        chi2_scores[(w1, w2)]   = chi2_stat
        p_values[(w1, w2)]      = p_val

    print(f"Total bigram types: {len(bigram_counts):,}")

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    filtered_bigrams = {
        bg:(co_occurences[bg]) for bg in bigram_counts
        if (p_values[bg] < 0.001) and (co_occurences[bg] > 50)
    }

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    sorted_bigrams = sorted(
        filtered_bigrams.items(),
        key=lambda x: x[1],
        reverse=True
    )
    print(sorted_bigrams)

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    import nltk
    nltk.download('wordnet')
    nltk.download('averaged_perceptron_tagger_eng')

    from nltk.tokenize import MWETokenizer
    from nltk.stem import WordNetLemmatizer
    from nltk.corpus import wordnet

    lemmatizer = WordNetLemmatizer()


    def get_wordnet_pos(word):
        tag = nltk.pos_tag([word])[0][1][0].upper()
        tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'R': wordnet.ADV}
        return tag_map.get(tag, wordnet.NOUN)  # default to NOUN

    collocations  = list(filtered_bigrams.keys())
    mwe_tokenizer = MWETokenizer(collocations, separator='_')

    def tokenize(abstract):
        """
        Full pipeline:
          1. Lowercase & clean
          2. Remove stopwords
          3. Merge known collocations (MWE)
          4. Lemmatize single-word tokens (collocations left as-is)
        """
        if not isinstance(abstract, str):
            return []
        text = abstract.lower()
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()

        words = [w for w in text.split() if w not in stop_words]

        # Merge collocations first
        tokens = mwe_tokenizer.tokenize(words)

        # Lemmatize only single-word tokens; leave collocation tokens (contain '_') as-is
        lemmatized = [
            token if '_' in token else lemmatizer.lemmatize(token, get_wordnet_pos(token))
            for token in tokens
        ]
        return lemmatized

    df["tokens"] = df["body_text"].apply(tokenize)

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    df["tokens"].head(5)

In [ ]:
if False:  # change to True to actually re-run this preprocessing step
    df.to_csv("tokenized_wikipage.csv",index = False)

## 2.4 Dataset stats — what we ended up with

Loading the cached outputs of the three preprocessing stages and computing the headline numbers.

In [ ]:
import pandas as pd

df_meta = pd.read_csv('politicians_with_communities.csv')
print(f'Politicians (nodes)     : {len(df_meta):,}')
print(f'Communities             : {df_meta["community"].nunique()}')
print()
print('Party breakdown:')
print(df_meta['party'].value_counts(dropna=False).to_string())
print()
print(f'Mean / median / max degree : {df_meta["degree"].mean():.1f} / {df_meta["degree"].median():.0f} / {df_meta["degree"].max()}')
print(f'Seeds vs neighbours        : {(df_meta["is_seed"]==True).sum()} / {(df_meta["is_seed"]==False).sum()}')

**Headline numbers (recycled from Project Assignment A):**

- **6,138 politicians** across **35 Louvain communities**
- **~19,700 mutual links** in the largest connected component
- **Party split:** 52% Democrat (3,190), 44% Republican (2,716), 4% Other/Independent (232)
- **Degree distribution:** mean 11, median 5, max ~1,000 — heavy-tailed, consistent with a few hub politicians (presidents, leaders) connected to many others.
- **Text corpus:** ~56 million characters after stripping footer sections; mean article length ~9,000 chars.

_HER SKLA VI KOMMENTERE PÅ: what these numbers suggest — e.g. how the Dem/Rep balance compares to the actual US political landscape, what the heavy degree tail implies, etc._

# 3 — Tools, theory and analysis

## 3.1 Network analysis

We model the data as an **undirected graph** $G = (V, E)$ where each node is a politician and each edge is a mutual Wikipedia link.

**Mutual links only.** A Wikipedia article often mentions other politicians in passing, but we only count an edge if *both* politicians' articles link to each other. This filters out one-way name-drops and gives a stronger signal of bilateral notability.

**Largest connected component.** Almost all 6,138 politicians end up in one giant component; the few isolates are excluded from the network analyses.

**Measures we compute and why:**

- **Degree distribution** — basic descriptor of network structure. We log-bin and plot it next to an Erdős–Rényi random graph with the same n and p. If our network has a much heavier tail than the random baseline, that's evidence of preferential attachment / hub structure (a few highly connected politicians).
- **Average shortest path length** — small-world test. If it's close to log(n) but the clustering coefficient is high relative to random, we're in small-world territory.
- **Centrality measures:**
  - *Degree centrality* — raw number of connections. Crude but interpretable.
  - *Closeness centrality* — `(N − 1) / Σ d(u, v)`. Penalises politicians that are far from everyone else.
  - *Eigenvector centrality* — a politician is central if they connect to other central politicians. Captures "prestige by association".
  - We compare degree vs. eigenvector and degree vs. closeness; high correlation suggests degree alone is a reasonable proxy.
- **Assortativity** — for each attribute (party, gender, prestigious_school, is_seed, degree) we compute Newman's assortativity coefficient `r`. `r > 0` means similar nodes connect to each other; `r < 0` means opposites attract; `r ≈ 0` means the attribute is unrelated to link structure.
- **Louvain community detection** — greedy modularity optimisation. Finds groups of nodes more densely connected internally than would be expected by chance. We then check whether these communities correspond to known categories (party, state).

_Write here why you chose these specific measures over alternatives (e.g. betweenness, PageRank, Girvan-Newman) and what hypothesis each one is testing._

In [ ]:
import networkx as nx
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter, defaultdict
import pandas as pd
import ast
import json
import random
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

df = pd.read_csv("politicians_network_nodes_filtered.csv")
print(f"Loaded {len(df)} politicians")
print(df.head())


### 1 — Load & filter

In [ ]:
def parse_connections(val):
    if isinstance(val, (list, tuple, set)):
        return set(val)
    try:
        if pd.isna(val):
            return set()
    except:
        pass
    if isinstance(val, str):
        try:
            return set(ast.literal_eval(val))
        except:
            return set()
    return set()

id_to_connections = {
    k: parse_connections(v)
    for k, v in zip(df["wikidata"], df["connections"])
}


In [ ]:
# Prestigous Schools
def get_prestigious_school_list(school):
    if pd.isna(school):
        return False
    school = school.lower()
    ELITE_POLITICAL_SCHOOLS = [
        # Ivy League (8)
        "harvard university", "harvard", "harvard college",
        "yale university", "yale", "yale college",
        "princeton university", "princeton",
        "columbia university", "columbia",
        "university of pennsylvania", "upenn", "penn",
        "brown university", "brown",
        "dartmouth college", "dartmouth",
        "cornell university", "cornell",
        
        # Ivy-equivalent private universities
        "stanford university", "stanford",
        "massachusetts institute of technology", "mit",
        "university of chicago", "uchicago",
        "duke university", "duke",
        "northwestern university", "northwestern",
        "johns hopkins university", "jhu",
        "california institute of technology", "caltech",
        "rice university", "rice",
        "vanderbilt university", "vanderbilt",
        "washington university in st. louis", "washu",
        "georgetown university", "georgetown",
        
        # Top liberal arts colleges
        "williams college", "williams",
        "amherst college", "amherst",
        "swarthmore college", "swarthmore",
        "wellesley college", "wellesley",
        "pomona college", "pomona",
        "middlebury college", "middlebury",
        "bowdoin college", "bowdoin",
        "claremont mckenna college", "cmc",
    ]
    return school in ELITE_POLITICAL_SCHOOLS

df["prestigious_school"] = df["education"].apply(get_prestigious_school_list)

In [ ]:
df["prestigious_school"].value_counts()

### 2 — Build graph

In [ ]:
# Mutual edges only — both politicians must link to each other
edges = {
    (person, other)
    for person, conns in id_to_connections.items()
    for other in conns
}


node_set = set(df["wikidata"])
G = nx.Graph()

for _, row in df.iterrows():
    G.add_node(row["wikidata"], **row.to_dict())

for (a, b) in edges:
    if a in node_set and b in node_set:
        G.add_edge(a, b)

largest_cc = max(nx.connected_components(G), key=len)
G_lcc = G.subgraph(largest_cc).copy()

print(f"\nFull graph — nodes: {G.number_of_nodes():,}  edges: {G.number_of_edges():,}")
print(f"Largest CC  — nodes: {G_lcc.number_of_nodes():,}  edges: {G_lcc.number_of_edges():,}")
print(f"Isolated nodes: {G.number_of_nodes() - len(largest_cc):,}")

# Save for reuse
with open("largest_cc.pickle", "wb") as f:
    pickle.dump(G_lcc, f, pickle.HIGHEST_PROTOCOL)
print("Saved → largest_cc.pickle")


In [ ]:
degree_map = dict(G_lcc.degree())

# Overwrite df["degree"] with actual network degrees and add new columns
df["degree"] = df["wikidata"].map(degree_map)

### 3 — Basic network statistics

In [ ]:
degrees      = [d for _, d in G_lcc.degree()]
mean_deg     = np.mean(degrees)
median_deg   = np.median(degrees)
n            = G_lcc.number_of_nodes()
m            = G_lcc.number_of_edges()
p_edge       = mean_deg / (n - 1)


print("=== Network Statistics ===")
print(f"  Nodes                  : {n:,}")
print(f"  Edges                  : {m:,}")
print(f"  Mean degree            : {mean_deg:.2f}")
print(f"  Median degree          : {median_deg:.1f}")
print(f"  Max degree             : {max(degrees)}")
print(f"  Min degree             : {min(degrees)}")

print("\nComputing average shortest path ")
avg_path = nx.average_shortest_path_length(G_lcc)
print(f"  Avg shortest path      : {avg_path:.4f}")



### 4 - Closeness and Eigen Centrality

In [ ]:
def closeness_centrality(G):
    nodes = list(G.nodes())
    N = G.number_of_nodes()
    closeness_dict = {}
    for node in nodes:
        lengths = nx.single_source_shortest_path_length(G, node)
        total_length = sum(lengths.values())
        closeness_dict[node] = (N-1)/total_length
    return closeness_dict


closeness_dict = closeness_centrality(G_lcc)

In [ ]:
sorted_items = sorted(closeness_dict.items(), key=lambda x: x[1], reverse=True)

# Top 5
top_5 = sorted_items[:5]

print("Top 5 nodes by closeness centrality:")
for node, centrality in top_5:
    print(f"Node {node}: {centrality:.4f}")

def eigenvector_centrality(G):
    return nx.eigenvector_centrality(G,max_iter=100,tol=10**(-6))

eigenvector_dict = eigenvector_centrality(G_lcc)

In [ ]:
# Plot degree vs centrality values

nodes = list(G_lcc.nodes())
deg_values = [degree_map[node] for node in nodes]
eigen_values = [eigenvector_dict[node] for node in nodes]
close_values = [closeness_dict[node] for node in nodes]

# Create figure
plt.figure(figsize=(14, 6))

# Degree vs Eigenvector
plt.subplot(1, 2, 1)
plt.scatter(deg_values, eigen_values, alpha=0.6, s=50, c='skyblue', edgecolors='navy')
plt.xlabel('Degree', fontsize=12)
plt.ylabel('Eigenvector Centrality', fontsize=12)
plt.title('Degree vs Eigenvector Centrality', fontsize=14)
plt.grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(deg_values, eigen_values, 1)
p = np.poly1d(z)
plt.plot(sorted(deg_values), p(sorted(deg_values)), "r--", alpha=0.8, linewidth=2)

# Calculate correlation
corr = np.corrcoef(deg_values, eigen_values)[0, 1]
plt.text(0.05, 0.95, f'Correlation: {corr:.3f}', 
         transform=plt.gca().transAxes, fontsize=12,
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

# Degree vs Closeness
plt.subplot(1, 2, 2)
plt.scatter(deg_values, close_values, alpha=0.6, s=50, c='salmon', edgecolors='darkred')
plt.xlabel('Degree', fontsize=12)
plt.ylabel('Closeness Centrality', fontsize=12)
plt.title('Degree vs Closeness Centrality', fontsize=14)
plt.grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(deg_values, close_values, 1)
p = np.poly1d(z)
plt.plot(sorted(deg_values), p(sorted(deg_values)), "r--", alpha=0.8, linewidth=2)

# Calculate correlation
corr = np.corrcoef(deg_values, close_values)[0, 1]
plt.text(0.05, 0.95, f'Correlation: {corr:.3f}', 
         transform=plt.gca().transAxes, fontsize=12,
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.show()

### 4 — Random network comparison

In [ ]:
# Erdos-Renyi random graph with same n and p
def generate_random_network(n, p, seed=None):
    """
    Generate a random network with n nodes where each possible edge
    is included independently with probability p.
    """
    if seed is not None:
        np.random.seed(seed)
    
    G_rand = nx.Graph()
    G_rand.add_nodes_from(range(n)) 
    
    # check every possible pair of nodes
    for i in range(n):
        for j in range(i+1, n):
            if np.random.uniform() < p:
                G_rand.add_edge(i, j)
    
    return G_rand

G_rand = generate_random_network(n=n,p=p_edge,seed = 20)
rand_lcc   = max(nx.connected_components(G_rand), key=len)
SG_rand    = G_rand.subgraph(rand_lcc).copy()
rand_deg   = [d for _, d in G_rand.degree()]
rand_path  = nx.average_shortest_path_length(SG_rand)

print("=== Real vs Random Network ===")
print(f"{'Metric':30s}  {'Real':>10s}  {'Random':>10s}")
print("-" * 55)
print(f"{'Nodes':30s}  {n:>10,}  {G_rand.number_of_nodes():>10,}")
print(f"{'Mean degree':30s}  {mean_deg:>10.2f}  {np.mean(rand_deg):>10.2f}")
print(f"{'Avg shortest path':30s}  {avg_path:>10.4f}  {rand_path:>10.4f}")

### 5 — Degree distribution

In [ ]:
degrees    = np.array([d for _, d in G_lcc.degree()])
rand_degs  = np.array([d for _, d in SG_rand.degree()])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Degree Distribution — Real vs Random Network", fontsize=14, fontweight="bold")

# Log-spaced bins for both
min_d = max(1, min(degrees.min(), rand_degs.min()))
max_d = max(degrees.max(), rand_degs.max())
bins  = np.logspace(np.log10(min_d), np.log10(max_d), 30)

for ax, data, color, title in [
    (axes[0], degrees,   "steelblue", "Real Network"),
    (axes[1], rand_degs, "salmon",    "Random Network"),
]:
    ax.hist(data, bins=bins, color=color, alpha=0.8,
            edgecolor="white", linewidth=0.5, log=True)
    ax.axvline(np.mean(data), color="red",    linestyle="--", linewidth=1.5,
               label=f"Mean = {np.mean(data):.1f}")
    ax.axvline(np.median(data), color="orange", linestyle="--", linewidth=1.5,
               label=f"Median = {np.median(data):.1f}")
    ax.set_xscale("log")
    ax.set_xlabel("Degree (log scale)"); ax.set_ylabel("Count (log scale)")
    ax.set_title(title); ax.legend(); ax.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.savefig("degree_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → degree_distribution.png")


In [ ]:
df["state"].value_counts()

### 8 — Assortativity

In [ ]:
def calculate_assortativity(G, attr):
    mixing  = defaultdict(int)
    

    for u, v in G.edges():
        cu = G.nodes[u][attr]
        cv = G.nodes[v][attr]

        mixing[(cu, cv)] += 1
        mixing[(cv, cu)] += 1

    m = G.number_of_edges()

    for key in mixing:
        mixing[key] /= (2 * m)

    a = defaultdict(float)
    for (i, j), val in mixing.items():
        a[i] += val

    sum_eii = sum(val for (i, j), val in mixing.items() if i == j)
    sum_aibi = sum(a[i] * a[i] for i in a)

    r = (sum_eii - sum_aibi) / (1 - sum_aibi + 1e-7)
    return r

results = {}
for attr in ["party", "gender","is_seed","prestigious_school"]:
    r = calculate_assortativity(G_lcc, attr)
    results[attr] = r
results["degree"] = nx.degree_assortativity_coefficient(G_lcc)

print("=== Assortativity coefficients ===")
for attr, r in results.items():
    bar = "█" * int(abs(r or 0) * 40)
    print(f"  {attr:15s}: {r:+.4f}  {bar}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
labels = list(results.keys())
values = [v if v is not None else 0 for v in results.values()]
colors = ["steelblue" if v >= 0 else "coral" for v in values]
ax.barh(labels, values, color=colors, alpha=0.8, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Assortativity coefficient")
ax.set_title("Assortativity — US Politicians Network")
plt.tight_layout()
plt.savefig("assortativity.png", dpi=150, bbox_inches="tight")
plt.show()


### 9 — Community detection (Louvain)

In [ ]:
print("Detecting Louvain communities...")
communities = nx.community.louvain_communities(G_lcc, seed=42)
partition   = {}
for i, comm in enumerate(communities):
    for node in comm:
        partition[node] = i

n_communities = len(communities)
print(f"Found {n_communities} communities")

for n_id in G_lcc.nodes():
    G_lcc.nodes[n_id]["community"] = partition.get(n_id, -1)
df["community"] = df["wikidata"].map(partition)

# Community sizes
sizes = Counter(partition.values())
print(f"  Largest community  : {max(sizes.values())} nodes")
print(f"  Smallest community : {min(sizes.values())} nodes")
print(f"  Median size        : {np.median(list(sizes.values())):.0f} nodes")

# Party composition per community
comm_party = (df.groupby(["community","party"])
                .size()
                .unstack(fill_value=0))
print("\nTop 5 communities by size:")
print(comm_party.loc[comm_party.sum(axis=1).nlargest(5).index])

df.to_csv("politicians_with_communities.csv", index=False)


communities = nx.community.louvain.louvain_communities(G_lcc)
partition = {}
for i, comm in enumerate(communities):
    for node in comm:
        partition[node] = i

# Map community → colour
cmap          = plt.cm.get_cmap("tab20", n_communities)
community_colors = {node: cmap(partition[node]) for node in G_lcc.nodes()}

print(f"Found {n_communities} communities")

# ── Use only top N nodes by degree for a readable plot ────────────────────────
MAX_NODES = 8000
top_nodes  = sorted(G_lcc.nodes(), key=lambda n: G_lcc.degree(n), reverse=True)[:MAX_NODES]
G_display  = G_lcc.subgraph(top_nodes).copy()

print(f"Computing layout for top {MAX_NODES} nodes by degree…")
pos = nx.kamada_kawai_layout(G_display)   # much better structure than spring

# Node properties
node_degrees = [G_display.degree(n) for n in G_display.nodes()]
max_deg      = max(node_degrees)
node_sizes   = [15 + 300 * (d / max_deg) ** 1.2 for d in node_degrees]
node_colors  = [community_colors[n] for n in G_display.nodes()]

# Edge alpha scaled by combined degree of endpoints — prominent edges stand out
edge_weights = []
for u, v in G_display.edges():
    combined = G_display.degree(u) + G_display.degree(v)
    edge_weights.append(combined)
max_w     = max(edge_weights) if edge_weights else 1
edge_alphas = [0.05 + 0.3 * (w / max_w) for w in edge_weights]

fig, ax = plt.subplots(figsize=(18, 18), facecolor="white")

# Draw edges with variable alpha
for (u, v), alpha in zip(G_display.edges(), edge_alphas):
    x = [pos[u][0], pos[v][0]]
    y = [pos[u][1], pos[v][1]]
    ax.plot(x, y, color="gray", alpha=alpha, linewidth=0.4, zorder=1)

# Draw nodes
nx.draw_networkx_nodes(G_display, pos,
                       node_color=node_colors,
                       node_size=node_sizes,
                       alpha=0.85,
                       linewidths=0.5,
                       edgecolors="white",
                       ax=ax)

# Label top 30 nodes only
top30  = sorted(G_display.nodes(), key=lambda n: G_display.degree(n), reverse=True)[:30]
labels = {n: G_display.nodes[n].get("name", n) for n in top30}
for node, label in labels.items():
    x, y = pos[node]
    ax.annotate(label, (x, y),
                fontsize=6.5, fontweight="bold",
                ha="center", va="bottom",
                xytext=(0, 6), textcoords="offset points",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.6, ec="none"))

# ── Second plot: full graph coloured by party ──────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(18, 18), facecolor="white")

PARTY_COLORS = {
    "Democrat":    "#2166ac",
    "Republican":  "#d6604d",
    "Other":       "#aaaaaa",
}

party_node_colors = [
    PARTY_COLORS.get(G_lcc.nodes[n].get("party"), "#dddddd")
    for n in G_display.nodes()
]

for (u, v), alpha in zip(G_display.edges(), edge_alphas):
    x = [pos[u][0], pos[v][0]]
    y = [pos[u][1], pos[v][1]]
    ax2.plot(x, y, color="gray", alpha=alpha * 0.6, linewidth=0.3, zorder=1)

nx.draw_networkx_nodes(G_display, pos,
                       node_color=party_node_colors,
                       node_size=node_sizes,
                       alpha=0.9,
                       linewidths=0.5,
                       edgecolors="white",
                       ax=ax2)

for node, label in labels.items():
    x, y = pos[node]
    ax2.annotate(label, (x, y),
                 fontsize=6.5, fontweight="bold",
                 ha="center", va="bottom",
                 xytext=(0, 6), textcoords="offset points",
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.6, ec="none"))

legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=10, label=p)
    for p, c in PARTY_COLORS.items() if p is not None
]
ax2.legend(handles=legend_elements, loc="upper left", fontsize=11, framealpha=0.8)
ax2.set_title(f"US Politicians Network — coloured by party\n"
              f"top {MAX_NODES} nodes by degree shown · "
              f"{G_lcc.number_of_nodes():,} total nodes · {G_lcc.number_of_edges():,} edges",
              fontsize=13, fontweight="bold")
ax2.axis("off")

plt.tight_layout()
fig.savefig("politicians_network_communities.png",  dpi=150, bbox_inches="tight")
fig2.savefig("politicians_network_party.png",       dpi=150, bbox_inches="tight")
plt.show()

### Output plots from the network analysis

![Degree distribution — real vs random](degree_distribution.png)

*Degree distribution on log-binned axes, compared to an Erdős–Rényi random graph with the same edge probability. The real network has a long right tail; the random baseline is sharply peaked around the mean.*

![Network coloured by Louvain community](politicians_network_communities.png)

*Force-directed layout of the largest connected component, with nodes coloured by Louvain community. The top 30 nodes by degree are labelled.*

![Network coloured by party](politicians_network_party.png)

*Same layout, coloured instead by party affiliation (blue = Democrat, red = Republican, grey = Other). Compare with the community plot — do the colours line up?*

## 3.2 Sentiment analysis

We use VADER to score every politician's article and ask five questions about the resulting distribution. Because compound scores are bounded in $[-1, 1]$ and pile up near zero, the data is **not normal**; we use non-parametric tests throughout:

- **Mann-Whitney U** for 2-group comparisons
- **Kruskal-Wallis** for 3+ group comparisons (with pairwise Mann-Whitney + Bonferroni if significant)
- **Spearman ρ** for rank correlations

**The five questions:**

1. **Q1 — Is Wikipedia neutral?** One-sample test against 0. If the mean compound score is significantly different from zero, the platform is not perfectly neutral on average.
2. **Q2 — Does sentiment differ by party?** Kruskal-Wallis across Democrat / Republican / Other.
3. **Q3 — Does sentiment differ by gender?** Mann-Whitney U on male vs female.
4. **Q4 — Does network centrality predict neutrality?** Spearman ρ between degree and `|sentiment|`. The hypothesis is that more-edited articles (higher-degree, more central politicians) might converge toward neutral.
5. **Q5 — Does sentiment vary with position tier?** Kruskal-Wallis across simplified tiers (President / VP / US Senator / etc.).

_Write here any caveats about VADER on long, formal text (it was tuned for short social posts)._

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pickle
from scipy import stats
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False,
                     "axes.spines.right": False})

df = pd.read_csv("tokenized_wikipage.csv")
print(f"Loaded {len(df)} politicians")
print(df[["name","party","gender","degree"]].head())


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if not text or not isinstance(text, str):
        return None
    return analyzer.polarity_scores(text)["compound"]

# Only compute if not already present
if "sentiment_score" not in df.columns or df["sentiment_score"].isna().all():
    df["sentiment_score"] = df["body_text"].apply(get_sentiment)

df["sentiment_abs"] = df["sentiment_score"].abs()

print(f"Articles with sentiment : {df['sentiment_score'].notna().sum()}")
print(f"Overall mean            : {df['sentiment_score'].mean():.4f}")
print(f"Overall std             : {df['sentiment_score'].std():.4f}")
print(f"Near-zero (|s| < 0.05)  : {(df['sentiment_abs'] < 0.05).sum()} "
      f"({(df['sentiment_abs'] < 0.05).mean()*100:.1f}%)")


### Statistical test choice

Sentiment scores are **continuous but not normally distributed** (VADER compounds are bounded [-1, 1] and tend to pile up near 0). We therefore use:

- **Mann-Whitney U** (2 groups) — non-parametric, no normality assumption
- **Kruskal-Wallis** (3+ groups) — non-parametric generalisation of ANOVA
- **Spearman ρ** (correlations) — non-parametric rank correlation

If Kruskal-Wallis is significant we run pairwise Mann-Whitney with Bonferroni correction.


### Q1 — Does Wikipedia neutrality hold?

If Wikipedia is truly neutral, the mean compound score should be near 0 and not significantly different from 0.

In [ ]:
scores = df["sentiment_score"].dropna()

# One-sample t-test against 0 (also report non-parametric Wilcoxon signed-rank)
t_stat, t_p = stats.ttest_1samp(scores, 0)
w_stat, w_p = stats.wilcoxon(scores)

print("=== Q1: Is overall sentiment neutral (≈0)? ===")
print(f"  Mean ± std          : {scores.mean():.4f} ± {scores.std():.4f}")
print(f"  Median              : {scores.median():.4f}")
print(f"  t-test vs 0         : t={t_stat:.3f}  p={t_p:.4e}")
print(f"  Wilcoxon signed-rank: W={w_stat:.1f}  p={w_p:.4e}")
print()
if t_p < 0.05:
    print("→ Sentiment is significantly different from 0 — Wikipedia is not perfectly neutral")
else:
    print("→ Cannot reject neutrality — mean sentiment is indistinguishable from 0")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(scores, bins=50, color="steelblue", alpha=0.8, edgecolor="white", linewidth=0.4)
ax.axvline(0,             color="black", linewidth=1.2, linestyle="--", label="Neutral (0)")
ax.axvline(scores.mean(), color="red",   linewidth=1.5, linestyle="-",
           label=f"Mean = {scores.mean():.3f}")
ax.set_xlabel("Compound sentiment score")
ax.set_ylabel("Count")
ax.set_title("Q1 — Overall sentiment distribution")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("q1_neutrality.png", dpi=150, bbox_inches="tight")
plt.show()


### Q2 — Does sentiment differ by party?

In [ ]:
parties = ["Democrat", "Republican", "Other"]
groups  = [df[df["party"] == p]["sentiment_score"].dropna() for p in parties]

# Kruskal-Wallis across all parties
h_stat, h_p = stats.kruskal(*groups)

print("=== Q2: Sentiment by party ===")
print(f"{'Party':15s}  {'n':>5}  {'mean':>7}  {'median':>8}  {'std':>6}")
for p, g in zip(parties, groups):
    print(f"  {p:13s}  {len(g):5d}  {g.mean():7.4f}  {g.median():8.4f}  {g.std():6.4f}")
print(f"Kruskal-Wallis: H={h_stat:.3f}  p={h_p:.4e}")

# Pairwise Mann-Whitney with Bonferroni
pairs = [("Democrat","Republican"), ("Democrat","Other"), ("Republican","Other")]
alpha_bonf = 0.05 / len(pairs)
print(f"Pairwise Mann-Whitney (Bonferroni α={alpha_bonf:.4f}):")
for p1, p2 in pairs:
    g1 = df[df["party"] == p1]["sentiment_score"].dropna()
    g2 = df[df["party"] == p2]["sentiment_score"].dropna()
    u, p = stats.mannwhitneyu(g1, g2, alternative="two-sided")
    sig = "✓ significant" if p < alpha_bonf else "✗ not significant"
    print(f"  {p1} vs {p2:12s}: U={u:.0f}  p={p:.4e}  {sig}")

COLORS = {"Democrat": "#2166ac", "Republican": "#d6604d", "Other": "#aaaaaa"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for p, g, color in zip(parties, groups, COLORS.values()):
    axes[0].hist(g, bins=35, alpha=0.5, color=color, label=p, density=True)
axes[0].axvline(0, color="black", linewidth=1, linestyle="--")
axes[0].set_xlabel("Compound sentiment"); axes[0].set_ylabel("Density")
axes[0].set_title("Sentiment by party"); axes[0].legend(fontsize=8)

means   = [g.mean() for g in groups]
sems    = [g.sem()  for g in groups]
axes[1].barh(parties, means, xerr=sems, color=list(COLORS.values()),
             alpha=0.8, edgecolor="white", height=0.5,
             error_kw=dict(ecolor="black", capsize=4))
axes[1].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("Mean sentiment ± SE")
axes[1].set_title(f"Mean sentiment by party (Kruskal-Wallis p={h_p:.3e})")

plt.tight_layout()
plt.savefig("q2_party_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()


### Q3 — Does sentiment differ by gender?

In [ ]:
genders = ["male", "female"]
g_male  = df[df["gender"] == "male"]["sentiment_score"].dropna()
g_fem   = df[df["gender"] == "female"]["sentiment_score"].dropna()

u_stat, u_p = stats.mannwhitneyu(g_male, g_fem, alternative="two-sided")

print("=== Q3: Sentiment by gender ===")
print(f"{'Gender':10s}  {'n':>5}  {'mean':>7}  {'median':>8}  {'std':>6}")
for label, g in zip(genders, [g_male, g_fem]):
    print(f"  {label:8s}  {len(g):5d}  {g.mean():7.4f}  {g.median():8.4f}  {g.std():6.4f}")
print(f"Mann-Whitney U: U={u_stat:.0f}  p={u_p:.4e}")
if u_p < 0.05:
    print("→ Significant gender difference in sentiment")
else:
    print("→ No significant gender difference in sentiment")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for label, g, color in zip(genders, [g_male, g_fem], ["#2166ac","#d6604d"]):
    axes[0].hist(g, bins=35, alpha=0.5, color=color, label=label, density=True)
axes[0].axvline(0, color="black", linewidth=1, linestyle="--")
axes[0].set_xlabel("Compound sentiment"); axes[0].set_ylabel("Density")
axes[0].set_title("Sentiment by gender"); axes[0].legend(fontsize=9)

axes[1].boxplot([g_male, g_fem], labels=genders, patch_artist=True,
                medianprops=dict(color="white", linewidth=2),
                boxprops=dict(alpha=0.7))
boxes = axes[1].patches
for box, color in zip(boxes, ["#2166ac","#d6604d"]):
    box.set_facecolor(color)
axes[1].set_ylabel("Compound sentiment")
axes[1].set_title(f"Sentiment by gender (p={u_p:.3e})")
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")

plt.tight_layout()
plt.savefig("q3_gender_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()


### Q4 — Does network centrality predict neutrality?

Hypothesis: more central politicians (higher degree/PageRank) have more carefully edited, more neutral articles.

In [ ]:
from scipy.stats import spearmanr

valid = df[df["sentiment_score"].notna() & df["degree"].notna()].copy()

r_deg,  p_deg  = spearmanr(valid["degree"], valid["sentiment_score"])
print("=== Q4: Centrality vs sentiment ===")
print(f"  Spearman ρ(degree, sentiment)   = {r_deg:+.3f}  p={p_deg:.4e}")


if p_deg < 0.05:
    direction = "negative" if r_deg < 0 else "positive"
    print(f" Significant {direction} correlation: "
          f"{'higher degree - more neutral' if r_deg < 0 else 'higher degree - more extreme'}")
else:
    print(" No significant correlation between centrality and sentiment neutrality")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(valid["degree"], valid["sentiment_score"],
                alpha=0.25, s=8, color="steelblue")
# Trend line
z = np.polyfit(valid["degree"], valid["sentiment_score"], 1)
xr = np.linspace(valid["degree"].min(), valid["degree"].max(), 100)
axes[0].plot(xr, np.polyval(z, xr), color="red", linewidth=1.5)
axes[0].set_xlabel("Degree"); axes[0].set_ylabel("Sentiment")
axes[0].set_title(f"Degree vs sentiment (ρ={r_deg:+.3f}, p={p_deg:.3e})")
plt.show()


### Q5 — Does sentiment differ by position tier?

In [ ]:
def simplify_position(pos):
    if pos is None: return "Other"
    p = pos.lower()
    if "president of the united states" in p and "vice" not in p        and "advisor" not in p and "senior" not in p: return "President"
    if "vice president" in p:        return "Vice President"
    if "united states senator" in p or ("senator" in p and "state senator" not in p):
        return "US Senator"
    if "united states representative" in p or "member of the united states house" in p:
        return "US Representative"
    if "governor" in p and "lieutenant" not in p: return "Governor"
    if "secretary" in p:             return "Cabinet Secretary"
    if "ambassador" in p:            return "Ambassador"
    if "mayor" in p:                 return "Mayor"
    if "advisor" in p or "counsel" in p or "director" in p: return "Advisor/Staff"
    return "Other"

df["position_simple"] = df["position"].apply(simplify_position)

tiers  = ["President","Vice President","US Senator","US Representative",
          "Governor","Cabinet Secretary","Advisor/Staff","Other"]
groups = [df[df["position_simple"] == t]["sentiment_score"].dropna() for t in tiers]
groups_valid = [(t, g) for t, g in zip(tiers, groups) if len(g) >= 5]

h_stat, h_p = stats.kruskal(*[g for _, g in groups_valid])
print("=== Q5: Sentiment by position tier ===")
print(f"{'Position':25s}  {'n':>5}  {'mean':>7}  {'median':>8}")
for t, g in groups_valid:
    print(f"  {t:23s}  {len(g):5d}  {g.mean():7.4f}  {g.median():8.4f}")
print(f"Kruskal-Wallis: H={h_stat:.3f}  p={h_p:.4e}")

labels_v = [t for t, _ in groups_valid]
means_v  = [g.mean() for _, g in groups_valid]
sems_v   = [g.sem()  for _, g in groups_valid]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(labels_v, means_v, xerr=sems_v, color="steelblue", alpha=0.8,
        edgecolor="white", height=0.6,
        error_kw=dict(ecolor="black", capsize=3))
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Mean sentiment ± SE")
ax.set_title(f"Q5 — Sentiment by position tier(Kruskal-Wallis p={h_p:.3e})")
plt.tight_layout()
plt.savefig("q5_position_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()


### Summary

In [ ]:
print("=== Summary of findings ===")
print()
print("Q1 Neutrality  : mean sentiment is", 
      "≠ 0 (Wikipedia is not fully neutral)" if t_p < 0.05
      else "≈ 0 (Wikipedia appears neutral overall)")
print(f"               : mean={scores.mean():.4f}  p={t_p:.3e}")
print()
print("Q2 Party       : Kruskal-Wallis",
      f"significant (H={h_stat:.2f}, p={h_p:.3e})" if h_p < 0.05
      else f"not significant (p={h_p:.3e})")
print()
print("Q3 Gender      :", "significant difference" if u_p < 0.05
      else "no significant difference",
      f"(Mann-Whitney p={u_p:.3e})")
print()
print("Q4 Centrality  : Spearman ρ =", f"{r_deg:+.3f}",
      f"(p={p_deg:.3e})",
      "significant" if p_deg < 0.05 else "not significant")
print()
print(f"Q5 Position    : Kruskal-Wallis p={h_p:.3e}",
      "significant" if h_p < 0.05 else "not significant")


## 3.3 TF-IDF — distinctive vocabulary per party and per community

- Why TF-IDF over raw term frequency (down-weights ubiquitous terms like *senate*, *election*)
- The full stopword stack (auto-generated politician names + state names + Wikipedia boilerplate + cities)
- Document-level TF-IDF then group-aggregated (rather than concatenating all articles per group and computing once)
- Outputs: `tfidf_party.png`, `tfidf_communities.png`

# 4 — Discussion



### What went well?

_..._

### What is still missing? What could be improved? Why?

_..._